In [11]:
# Find ALL problem rows before loading
problems = []
for i, row in subs.iterrows():
    try:
        pd.DataFrame([row]).to_sql("subscriptions", con=engine, if_exists="append", index=False)
    except Exception as e:
        problems.append({'row': i, 'data': row.to_dict(), 'error': str(e)})

print(f"Total problem rows: {len(problems)}")
for p in problems[:10]:  # show first 10
    print(f"\nRow {p['row']}: {p['data']}")
    print(f"Error: {p['error']}")

Total problem rows: 11

Row 0: {'user_id': 1, 'plan_type': 'Premium', 'monthly_fee': 19.99, 'status': 'Active', 'churn_date': None}
Error: (pymysql.err.IntegrityError) (1062, "Duplicate entry '1' for key 'subscriptions.PRIMARY'")
[SQL: INSERT INTO subscriptions (user_id, plan_type, monthly_fee, status, churn_date) VALUES (%(user_id)s, %(plan_type)s, %(monthly_fee)s, %(status)s, %(churn_date)s)]
[parameters: {'user_id': 1, 'plan_type': 'Premium', 'monthly_fee': 19.99, 'status': 'Active', 'churn_date': None}]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

Row 1: {'user_id': 2, 'plan_type': 'Basic', 'monthly_fee': 9.99, 'status': 'Churned', 'churn_date': datetime.date(2025, 6, 28)}
Error: (pymysql.err.IntegrityError) (1062, "Duplicate entry '2' for key 'subscriptions.PRIMARY'")
[SQL: INSERT INTO subscriptions (user_id, plan_type, monthly_fee, status, churn_date) VALUES (%(user_id)s, %(plan_type)s, %(monthly_fee)s, %(status)s, %(churn_date)s)]
[parameters: {'user_id': 2, 'pl

In [12]:
# Step 1 — Clear the table first
with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
    conn.execute(text("TRUNCATE TABLE subscriptions"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
    conn.commit()
print("✅ Table cleared")

# Step 2 — Check count
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM subscriptions")).scalar()
print(f"Rows in subscriptions now: {count}")  # should be 0

# Step 3 — Load all at once
subs.to_sql("subscriptions", con=engine, if_exists="append", index=False, chunksize=5000, method="multi")
print(f"✅ subscriptions loaded — {len(subs):,} rows")

# Step 4 — Verify
with engine.connect() as conn:
    result = conn.execute(text("SELECT status, COUNT(*) FROM subscriptions GROUP BY status"))
    for row in result:
        print(f"  {row[0]:<15} {row[1]:>8,} rows")

✅ Table cleared
Rows in subscriptions now: 0
✅ subscriptions loaded — 10,000 rows
  Active             8,956 rows
  Churned            1,044 rows


In [13]:
# Load Sessions
sess = pd.read_csv(f"{CLEAN_DIR}/sessions.csv")
sess['session_date'] = pd.to_datetime(sess['session_date']).dt.date
sess['device_type']  = sess['device_type'].str.strip()
sess = sess[sess['user_id'].isin(users['user_id'])].copy()

sess.to_sql("sessions", con=engine, if_exists="append", index=False, chunksize=5000, method="multi")
print(f"✅ sessions loaded — {len(sess):,} rows")

# Final verification — all 3 tables
print("\n── Final Verification ────────────────────────")
with engine.connect() as conn:
    for tbl in ["users", "subscriptions", "sessions"]:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {tbl}")).scalar()
        print(f"  {tbl:<20} {count:>10,} rows")

✅ sessions loaded — 125,448 rows

── Final Verification ────────────────────────
  users                    10,000 rows
  subscriptions            10,000 rows
  sessions                125,448 rows
